In [1]:
import pandas as pd


# Nettoyage des données — FIFA World Cup 2026 Player Performance

In [3]:

pd.set_option('display.max_columns', 100)

df = pd.read_csv('fifa_world_cup_2026_player_performance.csv')
print("Dimensions du fichier brut :", df.shape)
df.head()


Dimensions du fichier brut : (54600, 75)


,player_id,player_name,age,nationality,team,jersey_number,position,height_cm,weight_kg,preferred_foot,club_name,market_value_eur,match_id,match_date,stadium,city,opponent_team,tournament_stage,match_result,goals_team,goals_opponent,minutes_played,goals,assists,shots,shots_on_target,expected_goals_xg,expected_assists_xa,key_passes,successful_passes,total_passes,pass_accuracy,dribbles_attempted,successful_dribbles,crosses,successful_crosses,tackles,interceptions,clearances,blocks,aerial_duels_won,aerial_duels_lost,recoveries,defensive_actions,fouls_committed,fouls_suffered,yellow_cards,red_cards,offsides,saves,save_percentage,punches,clean_sheet,goals_conceded,penalty_saves,distance_covered_km,sprint_distance_km,top_speed_kmh,accelerations,decelerations,stamina_score,player_rating,performance_score,offensive_contribution,defensive_contribution,possession_impact,pressure_resistance,creativity_score,consistency_score,clutch_performance_score,total_goals_tournament,total_assists_tournament,total_minutes_tournament,player_of_match_awards,tournament_rating
0,P00055,Rodri Fati,26,Spanish,Spain,3,Goalkeeper,195,75,Left,RB Salzburg,4384884,M00001,2026-07-10,Hard Rock Stadium,Miami,South Africa,Group Stage,W,1,0,72,0,0,0,0,0.00,0.00,0,15,26,0.59,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,4,0.83,0,1,0,0,7.8,0.7,26.5,13,23,81.9,5.6,50.9,3.3,48.2,1.1,44.2,55.9,42.0,51.8,0,0,242,0,5.8
1,P00070,Ansu Le Normand,19,Spanish,Spain,18,Midfielder,178,75,Right,Chelsea,4918927,M00001,2026-07-10,Hard Rock Stadium,Miami,South Africa,Group Stage,W,1,0,90,0,0,0,0,0.01,0.00,1,35,40,0.89,1,0,1,0,0,1,3,1,2,0,2,5,0,1,0,0,0,0,0.00,0,0,0,0,10.4,1.1,29.0,19,17,85.5,5.7,55.9,37.9,29.4,3.5,38.2,43.7,31.1,52.7,0,3,342,0,5.5
2,P00066,Gavi Ramos,18,Spanish,Spain,14,Midfielder,177,72,Left,AIK,125015698,M00001,2026-07-10,Hard Rock Stadium,Miami,South Africa,Group Stage,W,1,0,73,1,0,2,0,0.08,0.07,2,72,85,0.85,0,0,3,0,1,1,0,0,1,2,4,2,0,0,0,0,0,0,0.00,0,0,0,0,8.8,1.3,33.7,30,19,88.8,8.3,82.9,79.8,78.6,15.3,99.0,99.0,83.4,54.8,1,1,245,0,8.4
3,P00073,Pedro Cubarsi,20,Spanish,Spain,21,Forward,182,74,Right,PSV Eindhoven,11805512,M00001,2026-07-10,Hard Rock Stadium,Miami,South Africa,Group Stage,W,1,0,80,1,1,5,2,0.00,0.21,0,12,19,0.67,1,0,0,0,1,1,1,0,3,0,1,3,0,1,0,0,0,0,0.00,0,0,0,0,9.6,1.0,32.1,26,19,89.2,6.9,67.5,47.3,6.9,1.2,19.8,42.3,40.9,78.5,5,3,422,0,6.7
4,P00059,Alvaro Oyarzabal,23,Spanish,Spain,7,Defender,191,81,Left,Juventus,13325174,M00001,2026-07-10,Hard Rock Stadium,Miami,South Africa,Group Stage,W,1,0,79,0,0,1,0,0.00,0.00,1,33,44,0.76,2,1,0,0,1,3,0,2,1,2,4,6,0,0,0,0,0,0,0.00,0,0,0,0,7.5,0.7,30.5,23,18,73.6,5.7,55.4,33.0,75.6,6.2,44.1,33.5,60.0,56.6,0,0,440,0,5.7


## 1. Inventaire général



In [4]:
print("Nombre de lignes (joueur-match) :", len(df))
print("Nombre de joueurs uniques :", df['player_id'].nunique())
print("Nombre de matchs uniques  :", df['match_id'].nunique())
print("Nombre d'équipes          :", df['team'].nunique())
print("Nombre de colonnes        :", df.shape[1])


Nombre de lignes (joueur-match) : 54600
Nombre de joueurs uniques : 1248
Nombre de matchs uniques  : 1050
Nombre d'équipes          : 48
Nombre de colonnes        : 75


In [5]:
df.dtypes.value_counts()


int64      43
float64    18
str        14
Name: count, dtype: int64

## 2. Valeurs manquantes



In [6]:
na_counts = df.isna().sum()
na_counts = na_counts[na_counts > 0]
if na_counts.empty:
    print("Aucune valeur manquante détectée sur l'ensemble des colonnes.")
else:
    print(na_counts.sort_values(ascending=False))


Aucune valeur manquante détectée sur l'ensemble des colonnes.


## 3. Doublons


In [7]:
print("Lignes strictement dupliquées :", df.duplicated().sum())
print("Doublons (player_id, match_id)    :", df.duplicated(subset=['player_id', 'match_id']).sum())


Lignes strictement dupliquées : 0
Doublons (player_id, match_id)    : 0


## 4. Cohérence interne des variables "fixes" par joueur




In [8]:
fixed_cols = ['player_name', 'team', 'nationality', 'position', 'age', 'market_value_eur']
for col in fixed_cols:
    n_values = df.groupby('player_id')[col].nunique()
    n_incoherent = (n_values > 1).sum()
    print(f"{col:20s} -> joueurs avec plusieurs valeurs différentes : {n_incoherent}")


player_name          -> joueurs avec plusieurs valeurs différentes : 0
team                 -> joueurs avec plusieurs valeurs différentes : 0
nationality          -> joueurs avec plusieurs valeurs différentes : 0
position             -> joueurs avec plusieurs valeurs différentes : 0
age                  -> joueurs avec plusieurs valeurs différentes : 0
market_value_eur     -> joueurs avec plusieurs valeurs différentes : 0


**Constat** : pour les 1 248 joueurs, ces variables sont parfaitement stables. Aucune
incohérence d'identité.


## 6. Vérification des bornes logiques (valeurs aberrantes)



In [11]:
bounds = {
    'age': (15, 45),
    'minutes_played': (0, 120),
    'pass_accuracy': (0, 1),
    'save_percentage': (0, 1),
    'height_cm': (150, 210),
    'weight_kg': (50, 110),
    'market_value_eur': (0, 300_000_000),
}

for col, (lo, hi) in bounds.items():
    out_of_range = df[(df[col] < lo) | (df[col] > hi)]
    print(f"{col:20s} hors [{lo}, {hi}] : {len(out_of_range):4d}  "
          f"(min observé = {df[col].min()}, max observé = {df[col].max()})")


age                  hors [15, 45] :    0  (min observé = 17, max observé = 39)
minutes_played       hors [0, 120] :    0  (min observé = 0, max observé = 90)
pass_accuracy        hors [0, 1] :    0  (min observé = 0.42, max observé = 0.97)
save_percentage      hors [0, 1] :    0  (min observé = 0.0, max observé = 1.0)
height_cm            hors [150, 210] :    0  (min observé = 163, max observé = 200)
weight_kg            hors [50, 110] :    0  (min observé = 65, max observé = 87)
market_value_eur     hors [0, 300000000] :    0  (min observé = 528822, max observé = 200000000)


In [12]:
for col in ['goals', 'assists', 'shots', 'yellow_cards', 'red_cards']:
    print(f"{col:15s} min={df[col].min()}  max={df[col].max()}")


goals           min=0  max=4
assists         min=0  max=3
shots           min=0  max=11
yellow_cards    min=0  max=1
red_cards       min=0  max=1


## 7. Table de correspondance nationalité → confédération


In [14]:
confederation_map = {
    # UEFA
    'Spanish': 'UEFA', 'French': 'UEFA', 'German': 'UEFA', 'Italian': 'UEFA',
    'English': 'UEFA', 'Scottish': 'UEFA', 'Dutch': 'UEFA', 'Belgian': 'UEFA',
    'Portuguese': 'UEFA', 'Croatian': 'UEFA', 'Serbian': 'UEFA', 'Polish': 'UEFA',
    'Swiss': 'UEFA', 'Austrian': 'UEFA', 'Swedish': 'UEFA', 'Danish': 'UEFA',
    'Ukrainian': 'UEFA', 'Turkish': 'UEFA',
    # CONMEBOL
    'Argentine': 'CONMEBOL', 'Brazilian': 'CONMEBOL', 'Uruguayan': 'CONMEBOL',
    'Colombian': 'CONMEBOL', 'Chilean': 'CONMEBOL', 'Peruvian': 'CONMEBOL',
    'Ecuadorian': 'CONMEBOL',
    # CAF
    'Moroccan': 'CAF', 'Senegalese': 'CAF', 'Nigerian': 'CAF', 'Tunisian': 'CAF',
    'Algerian': 'CAF', 'Cameroonian': 'CAF', 'Ghanaian': 'CAF', 'South African': 'CAF',
    'Egyptian': 'CAF',
    # AFC
    'Japanese': 'AFC', 'South Korean': 'AFC', 'Saudi': 'AFC', 'Iranian': 'AFC',
    'Qatari': 'AFC', 'Iraqi': 'AFC', 'Uzbek': 'AFC', 'Australian': 'AFC',
    # CONCACAF
    'American': 'CONCACAF', 'Mexican': 'CONCACAF', 'Canadian': 'CONCACAF',
    'Costa Rican': 'CONCACAF', 'Jamaican': 'CONCACAF', 'Panamanian': 'CONCACAF',
    # OFC
    # (aucune nationalité OFC présente dans ce dataset - à vérifier)
}

df['confederation'] = df['nationality'].map(confederation_map)

missing_mapping = df.loc[df['confederation'].isna(), 'nationality'].unique()
print("Nationalités non mappées (à corriger si non vide) :", missing_mapping)
print()
print(df['confederation'].value_counts())


Nationalités non mappées (à corriger si non vide) : <StringArray>
[]
Length: 0, dtype: str

confederation
UEFA        20072
CAF         10140
AFC          9308
CONMEBOL     7852
CONCACAF     7228
Name: count, dtype: int64


## 8. Agrégation au niveau joueur

On construit la table d'analyse finale, à un joueur = une ligne, en recalculant les totaux
tournoi à partir des lignes de match (cf. décision prise en section 5) et en conservant les
variables fixes.


In [15]:
# Variables à sommer sur l'ensemble des matchs du joueur
sum_cols = [
    'minutes_played', 'goals', 'assists', 'shots', 'shots_on_target',
    'expected_goals_xg', 'expected_assists_xa', 'key_passes',
    'successful_passes', 'total_passes', 'dribbles_attempted', 'successful_dribbles',
    'crosses', 'successful_crosses', 'tackles', 'interceptions', 'clearances',
    'blocks', 'aerial_duels_won', 'aerial_duels_lost', 'recoveries',
    'defensive_actions', 'fouls_committed', 'fouls_suffered', 'yellow_cards',
    'red_cards', 'offsides', 'saves', 'clean_sheet', 'goals_conceded',
    'penalty_saves', 'player_of_match_awards',
]

# Variables à moyenner (indicateurs déjà normalisés / scores par match)
mean_cols = [
    'pass_accuracy', 'save_percentage', 'distance_covered_km', 'sprint_distance_km',
    'top_speed_kmh', 'stamina_score', 'player_rating', 'performance_score',
    'offensive_contribution', 'defensive_contribution', 'possession_impact',
    'pressure_resistance', 'creativity_score', 'consistency_score',
    'clutch_performance_score',
]

# Variables fixes (identiques pour toutes les lignes d'un joueur)
fixed_cols_keep = [
    'player_name', 'age', 'nationality', 'confederation', 'team', 'position',
    'height_cm', 'weight_kg', 'preferred_foot', 'club_name', 'market_value_eur',
]

players = df.groupby('player_id').agg(
    **{c: (c, 'sum') for c in sum_cols},
    **{c: (c, 'mean') for c in mean_cols},
    **{c: (c, 'first') for c in fixed_cols_keep},
    n_matches=('match_id', 'nunique'),
).reset_index()

print("Table joueur :", players.shape)
players.head()


Table joueur : (1248, 60)


,player_id,minutes_played,goals,assists,shots,shots_on_target,expected_goals_xg,expected_assists_xa,key_passes,successful_passes,total_passes,dribbles_attempted,successful_dribbles,crosses,successful_crosses,tackles,interceptions,clearances,blocks,aerial_duels_won,aerial_duels_lost,recoveries,defensive_actions,fouls_committed,fouls_suffered,yellow_cards,red_cards,offsides,saves,clean_sheet,goals_conceded,penalty_saves,player_of_match_awards,pass_accuracy,save_percentage,distance_covered_km,sprint_distance_km,top_speed_kmh,stamina_score,player_rating,performance_score,offensive_contribution,defensive_contribution,possession_impact,pressure_resistance,creativity_score,consistency_score,clutch_performance_score,player_name,age,nationality,confederation,team,position,height_cm,weight_kg,preferred_foot,club_name,market_value_eur,n_matches
0,P00001,953,0,0,0,0,0.00,0.00,4,164,232,7,0,3,0,0,5,14,0,15,3,15,19,12,9,1,0,1,30,2,16,0,0,0.731176,0.267941,2.620588,0.288235,10.779412,80.552941,2.329412,23.829412,9.376471,80.473529,0.800000,66.391176,30.291176,69.691176,57.294118,Kylian Griezmann,24,French,UEFA,France,Goalkeeper,191,81,Right,PSV Eindhoven,16574124,34
1,P00002,830,0,0,0,0,0.00,0.00,6,154,217,5,0,2,0,0,3,16,0,16,2,13,19,6,5,7,0,2,37,1,18,1,0,0.728824,0.185000,2.350000,0.261765,8.464706,80.194118,1.961765,20.323529,11.152941,97.435294,1.091176,83.988235,30.747059,87.241176,59.800000,Antoine Tchouameni,29,French,UEFA,France,Goalkeeper,190,84,Right,Anderlecht,53959475,34
2,P00003,922,0,0,0,0,0.00,0.00,1,150,234,3,1,3,0,0,4,16,0,17,4,23,20,9,6,2,0,3,41,2,15,1,0,0.708529,0.212353,2.550000,0.305882,9.438235,79.920588,2.038235,21.750000,11.205882,79.588235,0.850000,68.508824,33.802941,70.752941,54.729412,Ousmane Hernandez,24,French,UEFA,France,Goalkeeper,191,77,Right,Besiktas,4935397,34
3,P00004,1299,0,0,8,1,0.06,0.01,9,472,591,8,0,24,1,41,31,56,14,39,25,53,142,15,9,3,0,0,0,0,0,0,0,0.811765,0.000000,4.150000,0.497059,19.094118,81.894118,3.473529,35.861765,23.958824,54.952941,1.729412,47.935294,33.617647,49.800000,50.752941,Kylian Camavinga,27,French,UEFA,France,Defender,183,80,Right,Bayer Leverkusen,3795900,34
4,P00005,1298,0,2,7,0,0.18,0.20,12,873,1078,6,0,24,1,43,41,54,11,31,25,87,149,13,12,7,0,3,0,0,0,0,0,0.821765,0.000000,4.079412,0.450000,19.817647,85.014706,4.270588,43.147059,48.211765,94.000000,5.420588,82.552941,30.941176,87.211765,63.764706,Antoine Kounde,24,French,UEFA,France,Defender,188,79,Right,Borussia Dortmund,21832889,34


## 9. Contrôle qualité final sur la table agrégée


In [16]:
print("Valeurs manquantes dans la table agrégée :")
print(players.isna().sum().sum())
print()
print("Répartition par poste :")
print(players['position'].value_counts())
print()
print("Répartition par confédération :")
print(players['confederation'].value_counts(dropna=False))


Valeurs manquantes dans la table agrégée :
0

Répartition par poste :
position
Defender      432
Midfielder    384
Forward       288
Goalkeeper    144
Name: count, dtype: int64

Répartition par confédération :
confederation
UEFA        468
CAF         234
AFC         208
CONMEBOL    182
CONCACAF    156
Name: count, dtype: int64


## 10. Export des tables nettoyées




In [18]:
df.to_csv('matches_clean.csv', index=False)
players.to_csv('players_clean.csv', index=False)

print("Fichiers exportés avec succès dans le dossier data/.")


Fichiers exportés avec succès dans le dossier data/.
